In [5]:
import os
import re
import json
from pathlib import Path

import pandas as pd

In [29]:
main_folder = Path.cwd().parent
path_excel = f"{main_folder}/data/excels/dosen-luar-tekkom-cleaned.xlsx"

df = pd.read_excel(path_excel)

In [30]:
df.head()

,nama_dosen,pengampu_mata_kuliah,foto_diri,profil_dosen_sinta,publikasi_dosen,profil_dosen_undip
0,"Prof. Dr. Eng. Agus Setyawan, S.Si., M.Si.",Fisika,https://tekkom.undip.ac.id/wp-content/uploads/...,https://sinta.kemdiktisaintek.go.id/authors/pr...,https://scholar.google.com/citations?hl=id&use...,https://scholar.undip.ac.id/en/persons/agus-se...
1,"Prof. Dr. Eng. Eko Hidayanto, S.Si., M.Si., F....",Fisika,https://tekkom.undip.ac.id/wp-content/uploads/...,https://sinta.kemdiktisaintek.go.id/authors/pr...,https://scholar.google.com/citations?hl=id&use...,https://scholar.undip.ac.id/en/persons/eko-hid...
2,"Dr.Eng. Udi Harmoko, S.Si., M.Si.",Fisika,https://tekkom.undip.ac.id/wp-content/uploads/...,https://sinta.kemdiktisaintek.go.id/authors/pr...,https://scholar.google.com/citations?hl=id&use...,https://scholar.undip.ac.id/en/persons/udi-har...
3,"Dr. Redemtus Heru Tjahjana, S.Si., M.Si.",Aljabar Linear,https://tekkom.undip.ac.id/wp-content/uploads/...,https://sinta.kemdiktisaintek.go.id/authors/pr...,https://scholar.google.com/citations?hl=id&use...,https://scholar.undip.ac.id/en/persons/redemtu...
4,"Dinar Mutiara Kusumo Nugraheni, S.T., M.InfoTe...",Audit SI/TI,https://tekkom.undip.ac.id/wp-content/uploads/...,https://sinta.kemdiktisaintek.go.id/authors/pr...,https://scholar.google.com/citations?hl=id&use...,https://scholar.undip.ac.id/en/persons/dinar-m...


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   nama_dosen            15 non-null     object
 1   pengampu_mata_kuliah  15 non-null     object
 2   foto_diri             15 non-null     object
 3   profil_dosen_sinta    15 non-null     object
 4   publikasi_dosen       15 non-null     object
 5   profil_dosen_undip    15 non-null     object
dtypes: object(6)
memory usage: 852.0+ bytes


In [32]:
df.isna().sum()

nama_dosen              0
pengampu_mata_kuliah    0
foto_diri               0
profil_dosen_sinta      0
publikasi_dosen         0
profil_dosen_undip      0
dtype: int64

In [33]:
sampel = df["nama_dosen"].iloc[3]

gelar = {'Prof.', 'Dr.', 'Ir.', 'S.T.', 'M.T.', 'M.Sc.', 'M.Eng.', 'M.Kom.', 'S.Kom', 'S.Si', 'S.Si.', 'S.H', 'S.H.', 'Eng.', 'Dr.Eng.', 'Dra.', 'Drs.'}

def hapus_gelar(nama):
    sampel = nama.split(',')[0].split()
    sampel = [nama for nama in sampel if nama not in gelar]
    return ' '.join(sampel)

In [35]:
slug_smp = hapus_gelar(sampel)

def buat_slug(nama):
    if (nama == ""):
        return ""
    slug = slug_smp.lower().split(' ')
    return f"dosen-luar-{'-'.join(slug)}"

buat_slug(slug_smp)

'dosen-luar-redemtus-heru-tjahjana'

In [36]:
kategori = "dalam"
NAMA_PRODI = "Teknik Komputer"
sumber_file = "dosen-luar-tekkom-cleaned.ipynb"
dokumen = []

def tampilkan_pesan_kosong(kolom):
    return f"Data {kolom} tidak tersedia"

for idx, row in df.iterrows():
    nama_dosen = row['nama_dosen'] if row["nama_dosen"] != "" else tampilkan_pesan_kosong("nama dosen")
    pengampu_matkul = row['pengampu_mata_kuliah'] if row['pengampu_mata_kuliah'] != "" else tampilkan_pesan_kosong("pengampu mata kuliah")
    
    baris = [f"Nama: {nama_dosen}, Nama tanpa gelar: {hapus_gelar(nama_dosen)}, Kategori: Dosen {kategori}, Pengampu: {pengampu_matkul}"]
    teks = (f"{nama_dosen} adalah dosen {kategori} di Program Studi {NAMA_PRODI} yang mengampu mata kuliah {pengampu_matkul}.\n\n"
            + "\n".join(baris))

    meta = {"tipe": "profil_dosen", 
            "kategori_dosen": kategori, 
            "nama": nama_dosen,
            "sumber_file": sumber_file}
    
    for profile in ('foto_diri', 'profil_dosen_sinta', 'publikasi_dosen', 'profil_dosen_undip'):
        meta[profile] = row[profile]

    dokumen.append({"teks": teks, "meta": meta, "slug": f"dosen-{kategori}-{buat_slug(hapus_gelar(nama_dosen))}"})

In [37]:
with open(f"{main_folder}/data/jsonl/dosen_luar_tekkom.jsonl", "w", encoding="utf-8") as f:
    for d in dokumen:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

In [18]:
df_dos_luar = pd.read_excel(f"{main_folder}/data/excels/dosen-luar-tekkom-cleaned.xlsx")
df_dos_dalam = pd.read_excel(f"{main_folder}/data/excels/dosen-dalam-tekkom-cleaned.xlsx")

In [19]:
df_dos_luar.columns.to_list()

['nama_dosen',
 'pengampu_mata_kuliah',
 'foto_diri',
 'profil_dosen_sinta',
 'publikasi_dosen',
 'profil_dosen_undip']

In [17]:
df_dos_dalam.columns.to_list()

['nama_dosen_dalam_tekkom',
 'keahlian',
 'foto_profil',
 'profil_sinta',
 'publikasi',
 'profil_undip']

In [ ]:
def ubah_kolom_standar(kategori: str, df: pd.DataFrame) -> pd.DataFrame:
    kolom_khusus = 'keahlian' if kategori == 'dalam' else 'pengampu'
    df.columns = ['nama_dosen', kolom_khusus, 'foto_profil', 'profil_sinta', 'publikasi', 'profil_google_scholar']
    return df

